# Optical source selection

This diagnostic notebook compares four optical configurations for the
ET downscaling workflow:

1. Sentinel-2 at 20 m
2. HLS-S30 at 30 m
3. HLS-L30 at 30 m
4. HLS-S30 + HLS-L30 at 30 m

The comparison is performed before selecting the final optical source
or prediction-grid resolution.

The evaluation is organized in three stages:

1. Product availability and temporal complementarity.
2. Station × MODIS-period spatial coverage and common optical predictors.
3. ET-model performance under identical validation folds.

No optical source is selected a priori.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import ee
import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError(
        "Repository root not found. Expected pyproject.toml."
    )


REPO_ROOT = find_repo_root()
SRC_PATH = REPO_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("Repository:", REPO_ROOT)
print("Source path:", SRC_PATH)

Repository: C:\Users\User\Desktop\cristian\GEE\26-et-downscaling-fundacion
Source path: C:\Users\User\Desktop\cristian\GEE\26-et-downscaling-fundacion\src


In [2]:
EE_PROJECT = "ee-change"

ee.Initialize(project=EE_PROJECT)
ee.Number(1).getInfo()

print("Earth Engine initialized with project:", EE_PROJECT)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Earth Engine initialized with project: ee-change


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


In [3]:
from et_downscaling.hls import (
    build_hls_medoid,
    get_hls_collection,
)

from et_downscaling.modis import (
    build_modis_inputs,
    get_modis_period_end,
)

from et_downscaling.sentinel2 import (
    build_s2_medoid,
    get_sentinel2_collection,
)

In [4]:
modis_inputs = build_modis_inputs()

station_footprints = modis_inputs[
    "station_footprints"
]

analysis_geometry = (
    station_footprints
    .geometry()
)


s2_collection = (
    get_sentinel2_collection(
        station_footprints
    )
)


hls_collection = (
    get_hls_collection(
        station_footprints
    )
)


hls_s30_collection = (
    hls_collection
    .filter(
        ee.Filter.eq(
            "sensor",
            "S30",
        )
    )
)


hls_l30_collection = (
    hls_collection
    .filter(
        ee.Filter.eq(
            "sensor",
            "L30",
        )
    )
)


OPTICAL_CANDIDATES = {
    "S2": {
        "collection": s2_collection,
        "scale_m": 20,
        "medoid_builder": build_s2_medoid,
    },
    "HLS_S30": {
        "collection": hls_s30_collection,
        "scale_m": 30,
        "medoid_builder": build_hls_medoid,
    },
    "HLS_L30": {
        "collection": hls_l30_collection,
        "scale_m": 30,
        "medoid_builder": build_hls_medoid,
    },
    "HLS_COMBINED": {
        "collection": hls_collection,
        "scale_m": 30,
        "medoid_builder": build_hls_medoid,
    },
}

print("Optical candidates:")
for name, config in OPTICAL_CANDIDATES.items():
    print(
        name,
        "->",
        config["scale_m"],
        "m",
    )

Optical candidates:
S2 -> 20 m
HLS_S30 -> 30 m
HLS_L30 -> 30 m
HLS_COMBINED -> 30 m


In [5]:
availability_rows = []

for name, config in OPTICAL_CANDIDATES.items():
    collection = config["collection"]

    product_count = (
        collection
        .size()
        .getInfo()
    )

    distinct_dates = (
        collection
        .aggregate_array(
            "date_key"
        )
        .distinct()
        .size()
        .getInfo()
    )

    availability_rows.append(
        {
            "source": name,
            "scale_m": config["scale_m"],
            "products": product_count,
            "distinct_dates": distinct_dates,
        }
    )


availability_summary = pd.DataFrame(
    availability_rows
)

display(
    availability_summary
)

,source,scale_m,products,distinct_dates
0,S2,20,439,217
1,HLS_S30,30,3161,638
2,HLS_L30,30,216,108
3,HLS_COMBINED,30,3377,681


In [6]:
s30_dates = set(
    hls_s30_collection
    .aggregate_array("date_key")
    .distinct()
    .getInfo()
)

l30_dates = set(
    hls_l30_collection
    .aggregate_array("date_key")
    .distinct()
    .getInfo()
)


hls_date_summary = pd.DataFrame(
    [
        {
            "metric": "S30 dates",
            "count": len(s30_dates),
        },
        {
            "metric": "L30 dates",
            "count": len(l30_dates),
        },
        {
            "metric": "Shared S30-L30 dates",
            "count": len(
                s30_dates & l30_dates
            ),
        },
        {
            "metric": "S30-only dates",
            "count": len(
                s30_dates - l30_dates
            ),
        },
        {
            "metric": "L30-only dates",
            "count": len(
                l30_dates - s30_dates
            ),
        },
        {
            "metric": "Combined unique dates",
            "count": len(
                s30_dates | l30_dates
            ),
        },
    ]
)

display(
    hls_date_summary
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


,metric,count
0,S30 dates,638
1,L30 dates,108
2,Shared S30-L30 dates,65
3,S30-only dates,573
4,L30-only dates,43
5,Combined unique dates,681


In [7]:
def summarize_station_availability(
    source_name: str,
    collection: ee.ImageCollection,
    scale_m: int,
) -> list[dict]:
    def summarize_feature(feature):
        feature = ee.Feature(feature)

        station_collection = (
            ee.ImageCollection(collection)
            .filterBounds(
                feature.geometry()
            )
        )

        return ee.Feature(
            None,
            {
                "source": source_name,
                "station": feature.get("station"),
                "station_id": feature.get("station_id"),
                "scale_m": scale_m,
                "products": station_collection.size(),
                "distinct_dates": (
                    ee.List(
                        station_collection.aggregate_array(
                            "date_key"
                        )
                    )
                    .distinct()
                    .size()
                ),
            },
        )

    result = ee.FeatureCollection(
        station_footprints.map(
            summarize_feature
        )
    )

    info = result.getInfo()

    return [
        feature["properties"]
        for feature in info["features"]
    ]


station_availability_rows = []

for source_name, config in OPTICAL_CANDIDATES.items():
    rows = summarize_station_availability(
        source_name=source_name,
        collection=config["collection"],
        scale_m=config["scale_m"],
    )

    station_availability_rows.extend(rows)


station_availability = pd.DataFrame(
    station_availability_rows
)

display(
    station_availability.sort_values(
        [
            "station",
            "source",
        ]
    )
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


,distinct_dates,products,scale_m,source,station,station_id
17,669,2985,30,HLS_COMBINED,Bananera,00000000000000000002
12,107,107,30,HLS_L30,Bananera,00000000000000000002
7,627,2878,30,HLS_S30,Bananera,00000000000000000002
2,217,218,20,S2,Bananera,00000000000000000002
19,670,2986,30,HLS_COMBINED,Bosque seco,00000000000000000004
14,108,108,30,HLS_L30,Bosque seco,00000000000000000004
9,627,2878,30,HLS_S30,Bosque seco,00000000000000000004
4,217,218,20,S2,Bosque seco,00000000000000000004
18,680,3376,30,HLS_COMBINED,Manglar,00000000000000000003
13,107,215,30,HLS_L30,Manglar,00000000000000000003


In [8]:
station_date_pivot = (
    station_availability
    .pivot(
        index="station",
        columns="source",
        values="distinct_dates",
    )
)

display(
    station_date_pivot
)

source,HLS_COMBINED,HLS_L30,HLS_S30,S2
station,,,,
Bananera,669,107,627,217
Bosque seco,670,108,627,217
Manglar,680,107,638,217
Palma,669,107,627,217
Pastos limpios,669,107,627,217


In [9]:
test_station = (
    station_footprints
    .filter(
        ee.Filter.eq(
            "station",
            "Bananera",
        )
    )
    .first()
)

test_geometry = (
    ee.Feature(
        test_station
    )
    .geometry()
)


hls_s30_test = (
    hls_s30_collection
    .filterBounds(
        test_geometry
    )
)


sample_hls_metadata = (
    hls_s30_test
    .limit(20)
    .map(
        lambda image: ee.Feature(
            None,
            {
                "system_index":
                    image.get(
                        "system:index"
                    ),

                "system_date":
                    image.date().format(
                        "yyyy-MM-dd HH:mm:ss"
                    ),

                "date_key":
                    image.get(
                        "date_key"
                    ),

                "sensing_time":
                    image.get(
                        "SENSING_TIME"
                    ),

                "mgrs_tile":
                    image.get(
                        "MGRS_TILE_ID"
                    ),

                "product_uri":
                    image.get(
                        "PRODUCT_URI"
                    ),
            },
        )
    )
)

sample_info = (
    sample_hls_metadata
    .getInfo()
)

sample_rows = [
    feature["properties"]
    for feature in sample_info["features"]
]

sample_metadata_df = (
    pd.DataFrame(
        sample_rows
    )
)

display(
    sample_metadata_df
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


,date_key,mgrs_tile,product_uri,sensing_time,system_date,system_index
0,2021-01-03,18PWS,S2A_MSIL1C_20210103T152641_N0209_R025_T18PWS_2...,None,2021-01-03 15:26:41,1_T18PWS_20210103T152641
1,2021-01-06,18PWS,S2A_MSIL1C_20210106T153621_N0209_R068_T18PWS_2...,None,2021-01-06 15:36:21,1_T18PWS_20210106T153621
2,2021-01-08,18PWS,S2B_MSIL1C_20210108T152639_N0209_R025_T18PWS_2...,None,2021-01-08 15:26:39,1_T18PWS_20210108T152639
3,2021-01-11,18PWS,S2B_MSIL1C_20210111T153619_N0209_R068_T18PWS_2...,None,2021-01-11 15:36:19,1_T18PWS_20210111T153619
4,2021-01-13,18PWS,S2A_MSIL1C_20210113T152641_N0209_R025_T18PWS_2...,None,2021-01-13 15:26:41,1_T18PWS_20210113T152641
5,2021-01-16,18PWS,S2A_MSIL1C_20210116T153621_N0209_R068_T18PWS_2...,None,2021-01-16 15:36:21,1_T18PWS_20210116T153621
6,2021-01-18,18PWS,S2B_MSIL1C_20210118T152639_N0209_R025_T18PWS_2...,None,2021-01-18 15:26:39,1_T18PWS_20210118T152639
7,2021-01-21,18PWS,S2B_MSIL1C_20210121T153619_N0209_R068_T18PWS_2...,None,2021-01-21 15:36:19,1_T18PWS_20210121T153619
8,2021-01-23,18PWS,S2A_MSIL1C_20210123T152641_N0209_R025_T18PWS_2...,None,2021-01-23 15:26:41,1_T18PWS_20210123T152641
9,2021-01-26,18PWS,S2A_MSIL1C_20210126T153621_N0209_R068_T18PWS_2...,None,2021-01-26 15:36:21,1_T18PWS_20210126T153621


In [11]:
first_hls_image = ee.Image(
    hls_s30_test.first()
)

property_names = (
    first_hls_image
    .propertyNames()
    .getInfo()
)

date_property_check = pd.DataFrame(
    [
        {
            "property": "system:time_start",
            "available": (
                "system:time_start"
                in property_names
            ),
        },
        {
            "property": "SENSING_TIME",
            "available": (
                "SENSING_TIME"
                in property_names
            ),
        },
        {
            "property": "PRODUCT_URI",
            "available": (
                "PRODUCT_URI"
                in property_names
            ),
        },
        {
            "property": "MGRS_TILE_ID",
            "available": (
                "MGRS_TILE_ID"
                in property_names
            ),
        },
    ]
)

display(date_property_check)

,property,available
0,system:time_start,True
1,SENSING_TIME,False
2,PRODUCT_URI,True
3,MGRS_TILE_ID,True


In [12]:
tile_histogram = (
    hls_s30_test
    .aggregate_histogram(
        "MGRS_TILE_ID"
    )
    .getInfo()
)

tile_table = (
    pd.DataFrame(
        [
            {
                "mgrs_tile": tile,
                "products": count,
            }
            for tile, count
            in tile_histogram.items()
        ]
    )
    .sort_values(
        "products",
        ascending=False,
    )
)

display(
    tile_table
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


,mgrs_tile,products
21,18PWS,275
16,01WCR,159
19,01WCU,154
18,01WCT,141
14,01WCP,138
9,01UBT,128
23,60UXC,124
20,01WCV,117
11,01VCL,115
26,60WWE,114


In [13]:
from et_downscaling.config import (
    START_DATE,
    END_DATE,
)


s2_raw_test = (
    ee.ImageCollection(
        "COPERNICUS/S2_SR_HARMONIZED"
    )
    .filterBounds(
        test_geometry
    )
    .filterDate(
        START_DATE,
        END_DATE,
    )
)


s2_linked_test = (
    s2_collection
    .filterBounds(
        test_geometry
    )
)


s2_collection_check = pd.DataFrame(
    [
        {
            "collection": "S2 raw",
            "products": (
                s2_raw_test
                .size()
                .getInfo()
            ),
            "distinct_dates": (
                s2_raw_test
                .aggregate_array(
                    "system:time_start"
                )
                .distinct()
                .size()
                .getInfo()
            ),
        },
        {
            "collection": "S2 linked",
            "products": (
                s2_linked_test
                .size()
                .getInfo()
            ),
            "distinct_dates": (
                s2_linked_test
                .aggregate_array(
                    "date_key"
                )
                .distinct()
                .size()
                .getInfo()
            ),
        },
    ]
)

display(
    s2_collection_check
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


,collection,products,distinct_dates
0,S2 raw,218,218
1,S2 linked,218,217


In [14]:
s2_tile_histogram = (
    s2_raw_test
    .aggregate_histogram(
        "MGRS_TILE"
    )
    .getInfo()
)

s2_tile_table = (
    pd.DataFrame(
        [
            {
                "mgrs_tile": tile,
                "products": count,
            }
            for tile, count
            in s2_tile_histogram.items()
        ]
    )
    .sort_values(
        "products",
        ascending=False,
    )
)

display(s2_tile_table)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


,mgrs_tile,products
0,18PWS,218


In [15]:
local_mgrs_tiles = list(
    s2_tile_histogram.keys()
)

hls_s30_local_tiles = (
    hls_s30_collection
    .filter(
        ee.Filter.inList(
            "MGRS_TILE_ID",
            local_mgrs_tiles,
        )
    )
)

local_hls_summary = pd.DataFrame(
    [
        {
            "source": "HLS_S30_all_filterBounds",
            "products": (
                hls_s30_test
                .size()
                .getInfo()
            ),
            "distinct_dates": (
                hls_s30_test
                .aggregate_array("date_key")
                .distinct()
                .size()
                .getInfo()
            ),
        },
        {
            "source": "HLS_S30_local_MGRS",
            "products": (
                hls_s30_local_tiles
                .size()
                .getInfo()
            ),
            "distinct_dates": (
                hls_s30_local_tiles
                .aggregate_array("date_key")
                .distinct()
                .size()
                .getInfo()
            ),
        },
    ]
)

display(local_hls_summary)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


,source,products,distinct_dates
0,HLS_S30_all_filterBounds,2878,627
1,HLS_S30_local_MGRS,276,276


In [16]:
from et_downscaling.hls import (
    prepare_hls_s30,
)


def add_valid_pixel_count(image):
    image = ee.Image(image)

    prepared = prepare_hls_s30(
        image
    )

    valid_mask = (
        prepared
        .select("Red")
        .mask()
        .rename("valid")
    )

    valid_count = (
        valid_mask
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=test_geometry,
            scale=30,
            maxPixels=1e6,
        )
        .get("valid")
    )

    return (
        image
        .set(
            "valid_pixels_at_station",
            valid_count,
        )
    )


hls_s30_local_checked = (
    hls_s30_local_tiles
    .map(
        add_valid_pixel_count
    )
)


hls_local_valid_summary = pd.DataFrame(
    [
        {
            "metric": "local tile products",
            "count": (
                hls_s30_local_checked
                .size()
                .getInfo()
            ),
        },
        {
            "metric": "products with valid pixels",
            "count": (
                hls_s30_local_checked
                .filter(
                    ee.Filter.gt(
                        "valid_pixels_at_station",
                        0,
                    )
                )
                .size()
                .getInfo()
            ),
        },
        {
            "metric": "dates with valid pixels",
            "count": (
                hls_s30_local_checked
                .filter(
                    ee.Filter.gt(
                        "valid_pixels_at_station",
                        0,
                    )
                )
                .aggregate_array(
                    "date_key"
                )
                .distinct()
                .size()
                .getInfo()
            ),
        },
    ]
)

display(
    hls_local_valid_summary
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


,metric,count
0,local tile products,276
1,products with valid pixels,113
2,dates with valid pixels,113


In [17]:
def add_hls_mask_diagnostics(image):
    image = ee.Image(image)

    # --------------------------------------------------------
    # 1. Raw spectral data availability
    # --------------------------------------------------------

    raw_spectral = (
        image
        .select(
            [
                "B2",
                "B3",
                "B4",
                "B8A",
                "B11",
                "B12",
            ]
        )
    )

    raw_valid = (
        raw_spectral
        .mask()
        .reduce(
            ee.Reducer.min()
        )
        .rename(
            "raw_valid"
        )
    )

    # --------------------------------------------------------
    # 2. HLS Fmask clear condition
    # --------------------------------------------------------

    fmask = image.select(
        "Fmask"
    )

    no_cloud = (
        fmask
        .bitwiseAnd(1 << 1)
        .eq(0)
    )

    no_adjacent = (
        fmask
        .bitwiseAnd(1 << 2)
        .eq(0)
    )

    no_shadow = (
        fmask
        .bitwiseAnd(1 << 3)
        .eq(0)
    )

    no_snow = (
        fmask
        .bitwiseAnd(1 << 4)
        .eq(0)
    )

    clear = (
        no_cloud
        .And(no_adjacent)
        .And(no_shadow)
        .And(no_snow)
        .rename("clear")
    )

    raw_clear = (
        raw_valid
        .And(clear)
        .rename(
            "raw_clear"
        )
    )

    # --------------------------------------------------------
    # 3. Positive-reflectance condition used by current code
    # --------------------------------------------------------

    positive = (
        raw_spectral
        .gt(0)
        .reduce(
            ee.Reducer.min()
        )
        .rename(
            "positive"
        )
    )

    full_current_mask = (
        raw_clear
        .And(positive)
        .rename(
            "full_current"
        )
    )

    diagnostic_stack = (
        raw_valid
        .addBands(raw_clear)
        .addBands(full_current_mask)
    )

    counts = (
        diagnostic_stack
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=test_geometry,
            scale=30,
            maxPixels=1e6,
        )
    )

    return (
        image
        .set(
            {
                "raw_valid_pixels":
                    counts.get(
                        "raw_valid"
                    ),

                "raw_clear_pixels":
                    counts.get(
                        "raw_clear"
                    ),

                "full_current_pixels":
                    counts.get(
                        "full_current"
                    ),
            }
        )
    )


hls_s30_mask_diagnostics = (
    hls_s30_local_tiles
    .map(
        add_hls_mask_diagnostics
    )
)


mask_diagnostic_summary = pd.DataFrame(
    [
        {
            "stage": "Local MGRS products",
            "products": (
                hls_s30_mask_diagnostics
                .size()
                .getInfo()
            ),
        },
        {
            "stage": "Raw data over footprint",
            "products": (
                hls_s30_mask_diagnostics
                .filter(
                    ee.Filter.gt(
                        "raw_valid_pixels",
                        0,
                    )
                )
                .size()
                .getInfo()
            ),
        },
        {
            "stage": "Raw data + Fmask clear",
            "products": (
                hls_s30_mask_diagnostics
                .filter(
                    ee.Filter.gt(
                        "raw_clear_pixels",
                        0,
                    )
                )
                .size()
                .getInfo()
            ),
        },
        {
            "stage": "Current full HLS mask",
            "products": (
                hls_s30_mask_diagnostics
                .filter(
                    ee.Filter.gt(
                        "full_current_pixels",
                        0,
                    )
                )
                .size()
                .getInfo()
            ),
        },
    ]
)

display(
    mask_diagnostic_summary
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


,stage,products
0,Local MGRS products,276
1,Raw data over footprint,190
2,Raw data + Fmask clear,113
3,Current full HLS mask,113


In [18]:
# ============================================================
# Sentinel-2 usable observations over the test footprint
# ============================================================

from et_downscaling.sentinel2 import (
    prepare_sentinel2,
)


def add_s2_valid_pixel_count(image):
    image = ee.Image(image)

    prepared = prepare_sentinel2(
        image
    )

    valid_mask = (
        prepared
        .select("Red")
        .mask()
        .rename("valid")
    )

    valid_count = (
        valid_mask
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=test_geometry,
            scale=20,
            maxPixels=1e6,
        )
        .get("valid")
    )

    return image.set(
        "valid_pixels_at_station",
        valid_count,
    )


s2_checked = (
    s2_linked_test
    .map(
        add_s2_valid_pixel_count
    )
)


s2_valid_products = (
    s2_checked
    .filter(
        ee.Filter.gt(
            "valid_pixels_at_station",
            0,
        )
    )
)


# ============================================================
# HLS-S30 high-aerosol diagnostic
# ============================================================

def add_hls_high_aerosol_diagnostic(image):
    image = ee.Image(image)

    raw_spectral = (
        image
        .select(
            [
                "B2",
                "B3",
                "B4",
                "B8A",
                "B11",
                "B12",
            ]
        )
    )

    spectral_valid = (
        raw_spectral
        .mask()
        .reduce(
            ee.Reducer.min()
        )
    )

    fmask = image.select(
        "Fmask"
    )

    no_cloud = (
        fmask
        .bitwiseAnd(1 << 1)
        .eq(0)
    )

    no_adjacent = (
        fmask
        .bitwiseAnd(1 << 2)
        .eq(0)
    )

    no_shadow = (
        fmask
        .bitwiseAnd(1 << 3)
        .eq(0)
    )

    no_snow = (
        fmask
        .bitwiseAnd(1 << 4)
        .eq(0)
    )

    aerosol_level = (
        fmask
        .rightShift(6)
        .bitwiseAnd(3)
    )

    not_high_aerosol = (
        aerosol_level
        .neq(3)
    )

    clear_mask = (
        spectral_valid
        .And(no_cloud)
        .And(no_adjacent)
        .And(no_shadow)
        .And(no_snow)
    )

    clear_no_high_aerosol = (
        clear_mask
        .And(
            not_high_aerosol
        )
        .rename(
            "valid"
        )
    )

    valid_count = (
        clear_no_high_aerosol
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=test_geometry,
            scale=30,
            maxPixels=1e6,
        )
        .get("valid")
    )

    return image.set(
        "valid_pixels_no_high_aerosol",
        valid_count,
    )


hls_s30_aerosol_checked = (
    hls_s30_local_tiles
    .map(
        add_hls_high_aerosol_diagnostic
    )
)


hls_s30_aerosol_valid = (
    hls_s30_aerosol_checked
    .filter(
        ee.Filter.gt(
            "valid_pixels_no_high_aerosol",
            0,
        )
    )
)


# ============================================================
# Comparable summary
# ============================================================

qa_comparison = pd.DataFrame(
    [
        {
            "source": "S2",
            "stage": "Raw products",
            "products": (
                s2_linked_test
                .size()
                .getInfo()
            ),
            "dates": (
                s2_linked_test
                .aggregate_array("date_key")
                .distinct()
                .size()
                .getInfo()
            ),
        },
        {
            "source": "S2",
            "stage": "Current QA",
            "products": (
                s2_valid_products
                .size()
                .getInfo()
            ),
            "dates": (
                s2_valid_products
                .aggregate_array("date_key")
                .distinct()
                .size()
                .getInfo()
            ),
        },
        {
            "source": "HLS_S30",
            "stage": "Raw local tile",
            "products": 276,
            "dates": 276,
        },
        {
            "source": "HLS_S30",
            "stage": "Raw data over footprint",
            "products": 190,
            "dates": None,
        },
        {
            "source": "HLS_S30",
            "stage": "Current Fmask clear",
            "products": 113,
            "dates": 113,
        },
        {
            "source": "HLS_S30",
            "stage": "Fmask + no high aerosol",
            "products": (
                hls_s30_aerosol_valid
                .size()
                .getInfo()
            ),
            "dates": (
                hls_s30_aerosol_valid
                .aggregate_array("date_key")
                .distinct()
                .size()
                .getInfo()
            ),
        },
    ]
)

display(
    qa_comparison
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


,source,stage,products,dates
0,S2,Raw products,218,217.0
1,S2,Current QA,162,161.0
2,HLS_S30,Raw local tile,276,276.0
3,HLS_S30,Raw data over footprint,190,NaN
4,HLS_S30,Current Fmask clear,113,113.0
5,HLS_S30,Fmask + no high aerosol,81,81.0


In [19]:
def get_s2_valid_dates(
    footprint,
):
    footprint = ee.Feature(
        footprint
    )

    geometry = footprint.geometry()

    collection = (
        s2_collection
        .filterBounds(
            geometry
        )
    )

    def add_valid_pixels(image):
        image = ee.Image(image)

        prepared = prepare_sentinel2(
            image
        )

        valid_pixels = (
            prepared
            .select("Red")
            .mask()
            .reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=geometry,
                scale=20,
                maxPixels=1e6,
            )
            .get("Red")
        )

        return image.set(
            "valid_pixels",
            valid_pixels,
        )

    valid_collection = (
        collection
        .map(
            add_valid_pixels
        )
        .filter(
            ee.Filter.gt(
                "valid_pixels",
                0,
            )
        )
    )

    return (
        valid_collection
        .aggregate_array(
            "date_key"
        )
        .distinct()
        .getInfo()
    )


def get_local_mgrs_tiles(
    footprint,
):
    footprint = ee.Feature(
        footprint
    )

    geometry = footprint.geometry()

    raw_s2 = (
        ee.ImageCollection(
            "COPERNICUS/S2_SR_HARMONIZED"
        )
        .filterBounds(
            geometry
        )
    )

    return (
        raw_s2
        .aggregate_array(
            "MGRS_TILE"
        )
        .distinct()
        .getInfo()
    )


def get_hls_valid_dates(
    footprint,
    collection,
    local_mgrs_tiles,
):
    footprint = ee.Feature(
        footprint
    )

    geometry = footprint.geometry()

    local_collection = (
        ee.ImageCollection(
            collection
        )
        .filter(
            ee.Filter.inList(
                "MGRS_TILE_ID",
                local_mgrs_tiles,
            )
        )
    )

    def add_valid_pixels(image):
        image = ee.Image(image)

        fmask = (
            image.select(
                "Fmask"
            )
        )

        no_cloud = (
            fmask
            .bitwiseAnd(
                1 << 1
            )
            .eq(0)
        )

        no_adjacent = (
            fmask
            .bitwiseAnd(
                1 << 2
            )
            .eq(0)
        )

        no_shadow = (
            fmask
            .bitwiseAnd(
                1 << 3
            )
            .eq(0)
        )

        no_snow = (
            fmask
            .bitwiseAnd(
                1 << 4
            )
            .eq(0)
        )

        aerosol_level = (
            fmask
            .rightShift(6)
            .bitwiseAnd(3)
        )

        no_high_aerosol = (
            aerosol_level
            .neq(3)
        )

        valid_mask = (
            image
            .select("B4")
            .mask()
            .And(
                no_cloud
            )
            .And(
                no_adjacent
            )
            .And(
                no_shadow
            )
            .And(
                no_snow
            )
            .And(
                no_high_aerosol
            )
            .rename(
                "valid"
            )
        )

        valid_pixels = (
            valid_mask
            .reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=geometry,
                scale=30,
                maxPixels=1e6,
            )
            .get("valid")
        )

        return image.set(
            "valid_pixels",
            valid_pixels,
        )

    valid_collection = (
        local_collection
        .map(
            add_valid_pixels
        )
        .filter(
            ee.Filter.gt(
                "valid_pixels",
                0,
            )
        )
    )

    return (
        valid_collection
        .aggregate_array(
            "date_key"
        )
        .distinct()
        .getInfo()
    )

In [20]:
station_features = (
    station_footprints
    .toList(
        station_footprints.size()
    )
)

station_count = (
    station_footprints
    .size()
    .getInfo()
)

optical_availability_rows = []


for index in range(station_count):

    footprint = ee.Feature(
        station_features.get(
            index
        )
    )

    station_name = (
        footprint
        .get("station")
        .getInfo()
    )

    print(
        "Processing:",
        station_name,
    )

    local_mgrs_tiles = (
        get_local_mgrs_tiles(
            footprint
        )
    )

    s2_dates = set(
        get_s2_valid_dates(
            footprint
        )
    )

    s30_dates = set(
        get_hls_valid_dates(
            footprint,
            hls_s30_collection,
            local_mgrs_tiles,
        )
    )

    l30_dates = set(
        get_hls_valid_dates(
            footprint,
            hls_l30_collection,
            local_mgrs_tiles,
        )
    )

    combined_dates = (
        s30_dates
        | l30_dates
    )

    optical_availability_rows.extend(
        [
            {
                "station": station_name,
                "source": "S2",
                "valid_dates": len(
                    s2_dates
                ),
            },
            {
                "station": station_name,
                "source": "HLS_S30",
                "valid_dates": len(
                    s30_dates
                ),
            },
            {
                "station": station_name,
                "source": "HLS_L30",
                "valid_dates": len(
                    l30_dates
                ),
            },
            {
                "station": station_name,
                "source": "HLS_COMBINED",
                "valid_dates": len(
                    combined_dates
                ),
            },
        ]
    )


optical_valid_dates = pd.DataFrame(
    optical_availability_rows
)

display(
    optical_valid_dates
)


optical_valid_dates_pivot = (
    optical_valid_dates
    .pivot(
        index="station",
        columns="source",
        values="valid_dates",
    )
)

display(
    optical_valid_dates_pivot
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Processing: Pastos limpios
Processing: Palma
Processing: Bananera
Processing: Manglar
Processing: Bosque seco


,station,source,valid_dates
0,Pastos limpios,S2,154
1,Pastos limpios,HLS_S30,90
2,Pastos limpios,HLS_L30,0
3,Pastos limpios,HLS_COMBINED,90
4,Palma,S2,158
5,Palma,HLS_S30,101
6,Palma,HLS_L30,0
7,Palma,HLS_COMBINED,101
8,Bananera,S2,161
9,Bananera,HLS_S30,81


source,HLS_COMBINED,HLS_L30,HLS_S30,S2
station,,,,
Bananera,81,0,81,161
Bosque seco,112,0,112,165
Manglar,125,0,125,158
Palma,101,0,101,158
Pastos limpios,90,0,90,154


In [21]:
# ============================================================
# Inspect HLS-L30 metadata and identifiers
# ============================================================

first_l30 = ee.Image(
    hls_l30_collection.first()
)

l30_property_names = (
    first_l30
    .propertyNames()
    .getInfo()
)

print("MGRS_TILE_ID available:", "MGRS_TILE_ID" in l30_property_names)
print("LANDSAT_PRODUCT_ID available:", "LANDSAT_PRODUCT_ID" in l30_property_names)
print()
print("Available properties:")
print(l30_property_names)


l30_sample = (
    hls_l30_collection
    .limit(20)
    .map(
        lambda image: ee.Feature(
            None,
            {
                "system_index":
                    image.get(
                        "system:index"
                    ),

                "system_date":
                    image.date().format(
                        "yyyy-MM-dd HH:mm:ss"
                    ),

                "mgrs_tile_id":
                    image.get(
                        "MGRS_TILE_ID"
                    ),

                "landsat_product_id":
                    image.get(
                        "LANDSAT_PRODUCT_ID"
                    ),

                "spatial_coverage":
                    image.get(
                        "SPATIAL_COVERAGE"
                    ),

                "cloud_coverage":
                    image.get(
                        "CLOUD_COVERAGE"
                    ),
            },
        )
    )
)


l30_sample_info = (
    l30_sample
    .getInfo()
)

l30_sample_df = pd.DataFrame(
    [
        feature["properties"]
        for feature
        in l30_sample_info["features"]
    ]
)

display(
    l30_sample_df
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


MGRS_TILE_ID available: False
LANDSAT_PRODUCT_ID available: True

Available properties:
['hls_sensor', 'system:version', 'system:id', 'sensor', 'date_key', 'system:index', 'B6_scale', 'B11_scale', 'B1_scale', 'HLS_PROCESSING_TIME', 'MEAN_SUN_AZIMUTH_ANGLE', 'B4_scale', 'SAA_scale', 'TIRS_SSM_MODEL', 'MEAN_VIEW_AZIMUTH_ANGLE', 'system:footprint', 'MEAN_SUN_ZENITH_ANGLE', 'CLOUD_COVERAGE', 'B2_scale', 'B7_scale', 'B10_scale', 'SPATIAL_COVERAGE', 'system:time_end', 'LANDSAT_PRODUCT_ID', 'B5_scale', 'SZA_scale', 'system:time_start', 'B9_scale', 'VZA_scale', 'ACCODE', 'B3_scale', 'MEAN_VIEW_ZENITH_ANGLE', 'NBAR_SOLAR_ZENITH', 'USGS_SOFTWARE', 'system:asset_size', 'TIRS_SSM_POSITION_STATUS', 'VAA_scale', 'system:bands', 'system:band_names']


,cloud_coverage,landsat_product_id,mgrs_tile_id,spatial_coverage,system_date,system_index
0,38,LC08_L1TP_009052_20210111_20210308_02_T1,None,100,2021-01-11 15:17:08,2_T18PWS_20210111T151708
1,24,LC08_L1TP_009052_20210111_20210308_02_T1,None,100,2021-01-11 15:17:08,2_T18PWT_20210111T151708
2,50,LC08_L1TP_009052_20210127_20210305_02_T1,None,100,2021-01-27 15:17:04,2_T18PWS_20210127T151704
3,99,LC08_L1TP_009052_20210127_20210305_02_T1,None,99,2021-01-27 15:17:04,2_T18PWT_20210127T151704
4,0,LC08_L1TP_009052_20210228_20210311_02_T1,None,100,2021-02-28 15:16:55,2_T18PWS_20210228T151655
5,1,LC08_L1TP_009052_20210228_20210311_02_T1,None,100,2021-02-28 15:16:55,2_T18PWT_20210228T151655
6,46,LC08_L1TP_009052_20210316_20210328_02_T1,None,100,2021-03-16 15:16:46,2_T18PWS_20210316T151646
7,63,LC08_L1TP_009052_20210316_20210328_02_T1,None,99,2021-03-16 15:16:46,2_T18PWT_20210316T151646
8,100,LC08_L1TP_009052_20210401_20210409_02_T1,None,100,2021-04-01 15:16:42,2_T18PWS_20210401T151642
9,87,LC08_L1TP_009052_20210401_20210409_02_T1,None,100,2021-04-01 15:16:42,2_T18PWT_20210401T151642


In [22]:
# ============================================================
# Derive HLS MGRS tile from system:index
# ============================================================

def add_hls_mgrs_tile_from_index(image):
    image = ee.Image(
        image
    )

    system_index = ee.String(
        image.get(
            "system:index"
        )
    )

    # Expected Earth Engine HLS index:
    #
    # 2_T18PWS_20210111T151708
    #
    # split("_")[1] -> T18PWS
    # slice(1)       -> 18PWS
    tile_token = ee.String(
        system_index
        .split("_")
        .get(1)
    )

    mgrs_tile = (
        tile_token
        .slice(1)
    )

    return (
        image
        .set(
            "derived_mgrs_tile",
            mgrs_tile,
        )
    )


hls_l30_with_tile = (
    hls_l30_collection
    .map(
        add_hls_mgrs_tile_from_index
    )
)


l30_local_tiles = (
    hls_l30_with_tile
    .filter(
        ee.Filter.inList(
            "derived_mgrs_tile",
            local_mgrs_tiles,
        )
    )
)


l30_local_summary = pd.DataFrame(
    [
        {
            "metric": "All L30 products",
            "count": (
                hls_l30_collection
                .size()
                .getInfo()
            ),
        },
        {
            "metric": "Local MGRS products",
            "count": (
                l30_local_tiles
                .size()
                .getInfo()
            ),
        },
        {
            "metric": "Local MGRS dates",
            "count": (
                l30_local_tiles
                .aggregate_array(
                    "date_key"
                )
                .distinct()
                .size()
                .getInfo()
            ),
        },
    ]
)

display(
    l30_local_summary
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


,metric,count
0,All L30 products,216
1,Local MGRS products,108
2,Local MGRS dates,108


In [23]:
# ============================================================
# HLS-L30 QA diagnostics over the test footprint
# ============================================================

from et_downscaling.hls import (
    prepare_hls_l30,
)


def add_l30_mask_diagnostics(image):
    image = ee.Image(
        image
    )

    # --------------------------------------------------------
    # Common optical bands
    #
    # B2 = Blue
    # B3 = Green
    # B4 = Red
    # B5 = NIR
    # B6 = SWIR1
    # B7 = SWIR2
    # --------------------------------------------------------

    raw_spectral = (
        image
        .select(
            [
                "B2",
                "B3",
                "B4",
                "B5",
                "B6",
                "B7",
            ]
        )
    )

    raw_valid = (
        raw_spectral
        .mask()
        .reduce(
            ee.Reducer.min()
        )
        .rename(
            "raw_valid"
        )
    )

    # --------------------------------------------------------
    # Fmask
    # --------------------------------------------------------

    fmask = (
        image.select(
            "Fmask"
        )
    )

    no_cloud = (
        fmask
        .bitwiseAnd(
            1 << 1
        )
        .eq(0)
    )

    no_adjacent = (
        fmask
        .bitwiseAnd(
            1 << 2
        )
        .eq(0)
    )

    no_shadow = (
        fmask
        .bitwiseAnd(
            1 << 3
        )
        .eq(0)
    )

    no_snow = (
        fmask
        .bitwiseAnd(
            1 << 4
        )
        .eq(0)
    )

    clear_mask = (
        raw_valid
        .And(no_cloud)
        .And(no_adjacent)
        .And(no_shadow)
        .And(no_snow)
        .rename(
            "clear"
        )
    )

    # --------------------------------------------------------
    # Aerosol
    # --------------------------------------------------------

    aerosol_level = (
        fmask
        .rightShift(6)
        .bitwiseAnd(3)
    )

    no_high_aerosol = (
        aerosol_level
        .neq(3)
    )

    clear_no_high_aerosol = (
        clear_mask
        .And(
            no_high_aerosol
        )
        .rename(
            "clear_no_high_aerosol"
        )
    )

    # --------------------------------------------------------
    # Current production HLS mask
    # --------------------------------------------------------

    prepared_current = (
        prepare_hls_l30(
            image
        )
    )

    current_valid = (
        prepared_current
        .select("Red")
        .mask()
        .rename(
            "current_valid"
        )
    )

    # --------------------------------------------------------
    # Count pixels over the MODIS footprint
    # --------------------------------------------------------

    diagnostic_stack = (
        raw_valid
        .addBands(
            clear_mask
        )
        .addBands(
            clear_no_high_aerosol
        )
        .addBands(
            current_valid
        )
    )

    counts = (
        diagnostic_stack
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=test_geometry,
            scale=30,
            maxPixels=1e6,
        )
    )

    return (
        image
        .set(
            {
                "raw_valid_pixels":
                    counts.get(
                        "raw_valid"
                    ),

                "clear_pixels":
                    counts.get(
                        "clear"
                    ),

                "clear_no_high_aerosol_pixels":
                    counts.get(
                        "clear_no_high_aerosol"
                    ),

                "current_valid_pixels":
                    counts.get(
                        "current_valid"
                    ),
            }
        )
    )


l30_mask_diagnostics = (
    l30_local_tiles
    .map(
        add_l30_mask_diagnostics
    )
)

In [24]:
# ============================================================
# HLS-L30 QA stage summary
# ============================================================

l30_qa_summary = pd.DataFrame(
    [
        {
            "stage": "Local MGRS products",
            "products": (
                l30_mask_diagnostics
                .size()
                .getInfo()
            ),
        },
        {
            "stage": "Raw data over footprint",
            "products": (
                l30_mask_diagnostics
                .filter(
                    ee.Filter.gt(
                        "raw_valid_pixels",
                        0,
                    )
                )
                .size()
                .getInfo()
            ),
        },
        {
            "stage": "Fmask clear",
            "products": (
                l30_mask_diagnostics
                .filter(
                    ee.Filter.gt(
                        "clear_pixels",
                        0,
                    )
                )
                .size()
                .getInfo()
            ),
        },
        {
            "stage": "Fmask + no high aerosol",
            "products": (
                l30_mask_diagnostics
                .filter(
                    ee.Filter.gt(
                        "clear_no_high_aerosol_pixels",
                        0,
                    )
                )
                .size()
                .getInfo()
            ),
        },
        {
            "stage": "Current prepare_hls_l30",
            "products": (
                l30_mask_diagnostics
                .filter(
                    ee.Filter.gt(
                        "current_valid_pixels",
                        0,
                    )
                )
                .size()
                .getInfo()
            ),
        },
    ]
)

display(
    l30_qa_summary
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


,stage,products
0,Local MGRS products,108
1,Raw data over footprint,107
2,Fmask clear,52
3,Fmask + no high aerosol,44
4,Current prepare_hls_l30,52


In [25]:
# ============================================================
# Common optical availability
# ============================================================

def add_hls_mgrs_tile(image):
    """
    Derive the HLS MGRS tile consistently for both S30 and L30
    from the Earth Engine system:index.

    Examples
    --------
    1_T18PWS_20210103T152641 -> 18PWS
    2_T18PWS_20210111T151708 -> 18PWS
    """
    image = ee.Image(image)

    system_index = ee.String(
        image.get("system:index")
    )

    tile_token = ee.String(
        system_index
        .split("_")
        .get(1)
    )

    mgrs_tile = (
        tile_token
        .slice(1)
    )

    return image.set(
        "hls_mgrs_tile",
        mgrs_tile,
    )


hls_s30_tiled = (
    hls_s30_collection
    .map(add_hls_mgrs_tile)
)

hls_l30_tiled = (
    hls_l30_collection
    .map(add_hls_mgrs_tile)
)


def get_local_mgrs_tiles(
    footprint,
):
    """
    Identify local MGRS tiles independently from Sentinel-2.
    """
    footprint = ee.Feature(footprint)
    geometry = footprint.geometry()

    local_s2 = (
        ee.ImageCollection(
            "COPERNICUS/S2_SR_HARMONIZED"
        )
        .filterBounds(geometry)
    )

    return (
        local_s2
        .aggregate_array("MGRS_TILE")
        .distinct()
        .getInfo()
    )


def get_s2_common_valid_dates(
    footprint,
):
    """
    Return dates containing at least one valid pixel for the
    six-band common optical comparison.

    Common bands:
    Blue, Green, Red, NIR, SWIR1, SWIR2.

    Sentinel-2 QA uses the current Cloud Score+ threshold.
    """
    footprint = ee.Feature(footprint)
    geometry = footprint.geometry()

    collection = (
        s2_collection
        .filterBounds(geometry)
    )

    def add_valid_pixels(image):
        image = ee.Image(image)

        # 10 m common bands
        bands_10m = (
            image
            .select(
                [
                    "B2",
                    "B3",
                    "B4",
                ]
            )
        )

        # 20 m common bands
        bands_20m = (
            image
            .select(
                [
                    "B8A",
                    "B11",
                    "B12",
                ]
            )
        )

        valid_10m = (
            bands_10m
            .mask()
            .reduce(
                ee.Reducer.min()
            )
        )

        valid_20m = (
            bands_20m
            .mask()
            .reduce(
                ee.Reducer.min()
            )
        )

        cloud_score_valid = (
            image
            .select("cs_cdf")
            .mask()
        )

        clear = (
            image
            .select("cs_cdf")
            .gte(0.60)
        )

        valid_mask = (
            valid_10m
            .And(valid_20m)
            .And(cloud_score_valid)
            .And(clear)
            .rename("valid")
        )

        valid_pixels = (
            valid_mask
            .reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=geometry,
                scale=20,
                maxPixels=1e6,
            )
            .get("valid")
        )

        return image.set(
            "common_valid_pixels",
            valid_pixels,
        )

    valid_collection = (
        collection
        .map(add_valid_pixels)
        .filter(
            ee.Filter.gt(
                "common_valid_pixels",
                0,
            )
        )
    )

    return set(
        valid_collection
        .aggregate_array("date_key")
        .distinct()
        .getInfo()
    )


def get_hls_common_valid_dates(
    footprint,
    collection,
    local_mgrs_tiles,
    sensor,
    exclude_high_aerosol=False,
):
    """
    Return HLS dates containing at least one valid common-band
    pixel over the footprint.
    """
    footprint = ee.Feature(footprint)
    geometry = footprint.geometry()

    local_collection = (
        ee.ImageCollection(collection)
        .filter(
            ee.Filter.inList(
                "hls_mgrs_tile",
                local_mgrs_tiles,
            )
        )
    )

    if sensor == "S30":
        common_source_bands = [
            "B2",
            "B3",
            "B4",
            "B8A",
            "B11",
            "B12",
        ]

    elif sensor == "L30":
        common_source_bands = [
            "B2",
            "B3",
            "B4",
            "B5",
            "B6",
            "B7",
        ]

    else:
        raise ValueError(
            f"Unsupported HLS sensor: {sensor}"
        )

    def add_valid_pixels(image):
        image = ee.Image(image)

        spectral_valid = (
            image
            .select(common_source_bands)
            .mask()
            .reduce(
                ee.Reducer.min()
            )
        )

        fmask = image.select("Fmask")

        no_cloud = (
            fmask
            .bitwiseAnd(1 << 1)
            .eq(0)
        )

        no_adjacent = (
            fmask
            .bitwiseAnd(1 << 2)
            .eq(0)
        )

        no_shadow = (
            fmask
            .bitwiseAnd(1 << 3)
            .eq(0)
        )

        no_snow = (
            fmask
            .bitwiseAnd(1 << 4)
            .eq(0)
        )

        valid_mask = (
            spectral_valid
            .And(no_cloud)
            .And(no_adjacent)
            .And(no_shadow)
            .And(no_snow)
        )

        if exclude_high_aerosol:
            aerosol_level = (
                fmask
                .rightShift(6)
                .bitwiseAnd(3)
            )

            valid_mask = (
                valid_mask
                .And(
                    aerosol_level.neq(3)
                )
            )

        valid_mask = (
            valid_mask
            .rename("valid")
        )

        valid_pixels = (
            valid_mask
            .reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=geometry,
                scale=30,
                maxPixels=1e6,
            )
            .get("valid")
        )

        return image.set(
            "common_valid_pixels",
            valid_pixels,
        )

    valid_collection = (
        local_collection
        .map(add_valid_pixels)
        .filter(
            ee.Filter.gt(
                "common_valid_pixels",
                0,
            )
        )
    )

    return set(
        valid_collection
        .aggregate_array("date_key")
        .distinct()
        .getInfo()
    )

In [26]:
# ============================================================
# Common-band availability by station
# ============================================================

station_list = (
    station_footprints
    .toList(
        station_footprints.size()
    )
)

station_count = (
    station_footprints
    .size()
    .getInfo()
)

availability_rows = []


for index in range(station_count):

    footprint = ee.Feature(
        station_list.get(index)
    )

    station_name = (
        footprint
        .get("station")
        .getInfo()
    )

    print(
        "Processing:",
        station_name,
    )

    local_tiles = (
        get_local_mgrs_tiles(
            footprint
        )
    )

    s2_dates = (
        get_s2_common_valid_dates(
            footprint
        )
    )

    s30_dates = (
        get_hls_common_valid_dates(
            footprint=footprint,
            collection=hls_s30_tiled,
            local_mgrs_tiles=local_tiles,
            sensor="S30",
            exclude_high_aerosol=True,
        )
    )

    l30_dates = (
        get_hls_common_valid_dates(
            footprint=footprint,
            collection=hls_l30_tiled,
            local_mgrs_tiles=local_tiles,
            sensor="L30",
            exclude_high_aerosol=False,
        )
    )

    # Sensitivity analysis only.
    l30_no_high_aerosol_dates = (
        get_hls_common_valid_dates(
            footprint=footprint,
            collection=hls_l30_tiled,
            local_mgrs_tiles=local_tiles,
            sensor="L30",
            exclude_high_aerosol=True,
        )
    )

    combined_dates = (
        s30_dates
        | l30_dates
    )

    availability_rows.append(
        {
            "station": station_name,
            "S2": len(s2_dates),
            "HLS_S30": len(s30_dates),
            "HLS_L30": len(l30_dates),
            "HLS_L30_no_high_aerosol": len(
                l30_no_high_aerosol_dates
            ),
            "HLS_COMBINED": len(
                combined_dates
            ),
            "L30_added_to_S30": len(
                l30_dates - s30_dates
            ),
            "S30_L30_shared": len(
                s30_dates & l30_dates
            ),
        }
    )


common_availability = pd.DataFrame(
    availability_rows
)

display(common_availability)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Processing: Pastos limpios
Processing: Palma
Processing: Bananera
Processing: Manglar
Processing: Bosque seco


,station,S2,HLS_S30,HLS_L30,HLS_L30_no_high_aerosol,HLS_COMBINED,L30_added_to_S30,S30_L30_shared
0,Pastos limpios,155,90,53,49,136,46,7
1,Palma,158,101,51,46,145,44,7
2,Bananera,161,81,52,44,126,45,7
3,Manglar,158,125,64,59,182,57,7
4,Bosque seco,165,112,61,57,164,52,9


In [27]:
# ============================================================
# MODIS 8-day periods
# ============================================================

from datetime import date, timedelta


modis_collection = ee.ImageCollection(
    modis_inputs["collection"]
)

period_millis = (
    modis_collection
    .aggregate_array(
        "system:time_start"
    )
    .getInfo()
)

period_starts = sorted(
    {
        pd.to_datetime(
            value,
            unit="ms",
            utc=True,
        ).date()
        for value in period_millis
    }
)


def get_period_end(
    period_start: date,
) -> date:
    regular_end = (
        period_start
        + timedelta(days=8)
    )

    next_year_start = date(
        period_start.year + 1,
        1,
        1,
    )

    return min(
        regular_end,
        next_year_start,
    )


periods = [
    (
        period_start,
        get_period_end(
            period_start
        ),
    )
    for period_start in period_starts
]

print(
    "MODIS periods:",
    len(periods),
)

print(
    "First period:",
    periods[0],
)

print(
    "Last period:",
    periods[-1],
)

assert len(periods) == 138

MODIS periods: 138
First period: (datetime.date(2021, 1, 1), datetime.date(2021, 1, 9))
Last period: (datetime.date(2023, 12, 27), datetime.date(2024, 1, 1))


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


In [28]:
# ============================================================
# Convert valid dates to MODIS-period availability
# ============================================================

def parse_date_set(
    date_strings,
):
    return {
        pd.to_datetime(
            value
        ).date()
        for value in date_strings
    }


def get_period_date_counts(
    valid_dates,
):
    valid_dates = parse_date_set(
        valid_dates
    )

    counts = []

    for (
        period_start,
        period_end,
    ) in periods:

        count = sum(
            period_start
            <= observation_date
            < period_end
            for observation_date
            in valid_dates
        )

        counts.append(
            count
        )

    return counts


def get_available_periods(
    valid_dates,
):
    counts = get_period_date_counts(
        valid_dates
    )

    return {
        index
        for index, count
        in enumerate(counts)
        if count > 0
    }

In [29]:
# ============================================================
# Optical availability by MODIS period
# ============================================================

station_list = (
    station_footprints
    .toList(
        station_footprints.size()
    )
)

station_count = (
    station_footprints
    .size()
    .getInfo()
)

period_availability_rows = []


for index in range(
    station_count
):

    footprint = ee.Feature(
        station_list.get(
            index
        )
    )

    station_name = (
        footprint
        .get("station")
        .getInfo()
    )

    print(
        "Processing:",
        station_name,
    )

    local_tiles = (
        get_local_mgrs_tiles(
            footprint
        )
    )

    # --------------------------------------------------------
    # Sentinel-2
    # --------------------------------------------------------

    s2_dates = (
        get_s2_common_valid_dates(
            footprint
        )
    )

    # --------------------------------------------------------
    # HLS-S30
    # --------------------------------------------------------

    s30_dates = (
        get_hls_common_valid_dates(
            footprint=footprint,
            collection=hls_s30_tiled,
            local_mgrs_tiles=local_tiles,
            sensor="S30",
            exclude_high_aerosol=True,
        )
    )

    # --------------------------------------------------------
    # HLS-L30
    # --------------------------------------------------------

    l30_dates = (
        get_hls_common_valid_dates(
            footprint=footprint,
            collection=hls_l30_tiled,
            local_mgrs_tiles=local_tiles,
            sensor="L30",
            exclude_high_aerosol=False,
        )
    )

    combined_dates = (
        s30_dates
        | l30_dates
    )

    # --------------------------------------------------------
    # Convert dates to MODIS periods
    # --------------------------------------------------------

    s2_periods = (
        get_available_periods(
            s2_dates
        )
    )

    s30_periods = (
        get_available_periods(
            s30_dates
        )
    )

    l30_periods = (
        get_available_periods(
            l30_dates
        )
    )

    combined_periods = (
        get_available_periods(
            combined_dates
        )
    )

    # Periods that become available only because L30 exists.
    l30_rescued_periods = (
        combined_periods
        - s30_periods
    )

    # Shared availability between the two main alternatives.
    s2_hls_shared_periods = (
        s2_periods
        & combined_periods
    )

    period_availability_rows.append(
        {
            "station":
                station_name,

            "S2_periods":
                len(
                    s2_periods
                ),

            "HLS_S30_periods":
                len(
                    s30_periods
                ),

            "HLS_L30_periods":
                len(
                    l30_periods
                ),

            "HLS_COMBINED_periods":
                len(
                    combined_periods
                ),

            "L30_rescued_periods":
                len(
                    l30_rescued_periods
                ),

            "S2_HLS_shared_periods":
                len(
                    s2_hls_shared_periods
                ),

            "S2_missing_periods":
                len(periods)
                - len(s2_periods),

            "HLS_COMBINED_missing_periods":
                len(periods)
                - len(combined_periods),
        }
    )


period_availability = pd.DataFrame(
    period_availability_rows
)

display(
    period_availability
)

Processing: Pastos limpios
Processing: Palma
Processing: Bananera
Processing: Manglar
Processing: Bosque seco


,station,S2_periods,HLS_S30_periods,HLS_L30_periods,HLS_COMBINED_periods,L30_rescued_periods,S2_HLS_shared_periods,S2_missing_periods,HLS_COMBINED_missing_periods
0,Pastos limpios,114,75,53,87,12,84,24,51
1,Palma,119,86,51,96,10,95,19,42
2,Bananera,118,69,52,87,18,83,20,51
3,Manglar,116,98,64,114,16,106,22,24
4,Bosque seco,120,91,61,103,12,98,18,35


In [30]:
# ============================================================
# Common optical valid masks
# ============================================================

from et_downscaling.config import (
    S2_CLEAR_THRESHOLD,
    S2_QA_BAND,
)


def build_s2_common_valid_mask(image):
    """
    Build the common-band Sentinel-2 valid mask at 20 m.

    Common comparison bands:
    Blue, Green, Red, NIR, SWIR1, SWIR2.

    The 10 m validity and Cloud Score+ masks are aggregated to
    the native 20 m B8A grid using a strict all-subpixels rule.
    """
    image = ee.Image(image)

    reference_projection = (
        image
        .select("B8A")
        .projection()
    )

    valid_10m_spectral = (
        image
        .select(
            [
                "B2",
                "B3",
                "B4",
            ]
        )
        .mask()
        .reduce(
            ee.Reducer.min()
        )
    )

    valid_qa = (
        image
        .select(S2_QA_BAND)
        .mask()
    )

    clear_10m = (
        image
        .select(S2_QA_BAND)
        .gte(S2_CLEAR_THRESHOLD)
    )

    valid_clear_10m = (
        valid_10m_spectral
        .And(valid_qa)
        .And(clear_10m)
        .unmask(0)
    )

    # Require all four nested 10 m pixels to be valid and clear.
    valid_clear_20m = (
        valid_clear_10m
        .reduceResolution(
            reducer=ee.Reducer.min(),
            maxPixels=4,
        )
        .reproject(
            reference_projection
        )
        .eq(1)
    )

    valid_native_20m = (
        image
        .select(
            [
                "B8A",
                "B11",
                "B12",
            ]
        )
        .mask()
        .reduce(
            ee.Reducer.min()
        )
        .reproject(
            reference_projection
        )
    )

    return (
        valid_clear_20m
        .And(valid_native_20m)
        .rename("valid")
        .uint8()
    )


def build_hls_common_valid_mask(
    image,
    sensor,
    exclude_high_aerosol=False,
):
    """
    Build an HLS common-band valid mask at 30 m.
    """
    image = ee.Image(image)

    if sensor == "S30":
        source_bands = [
            "B2",
            "B3",
            "B4",
            "B8A",
            "B11",
            "B12",
        ]

    elif sensor == "L30":
        source_bands = [
            "B2",
            "B3",
            "B4",
            "B5",
            "B6",
            "B7",
        ]

    else:
        raise ValueError(
            f"Unsupported HLS sensor: {sensor}"
        )

    spectral_valid = (
        image
        .select(source_bands)
        .mask()
        .reduce(
            ee.Reducer.min()
        )
    )

    fmask = image.select("Fmask")

    no_cloud = (
        fmask
        .bitwiseAnd(1 << 1)
        .eq(0)
    )

    no_adjacent = (
        fmask
        .bitwiseAnd(1 << 2)
        .eq(0)
    )

    no_shadow = (
        fmask
        .bitwiseAnd(1 << 3)
        .eq(0)
    )

    no_snow = (
        fmask
        .bitwiseAnd(1 << 4)
        .eq(0)
    )

    valid = (
        spectral_valid
        .And(no_cloud)
        .And(no_adjacent)
        .And(no_shadow)
        .And(no_snow)
    )

    if exclude_high_aerosol:
        aerosol_level = (
            fmask
            .rightShift(6)
            .bitwiseAnd(3)
        )

        valid = (
            valid
            .And(
                aerosol_level.neq(3)
            )
        )

    return (
        valid
        .unmask(0)
        .rename("valid")
        .uint8()
    )

In [31]:
# ============================================================
# Union coverage inside a MODIS footprint
# ============================================================

def build_valid_union(
    collection,
    mask_function,
):
    """
    Return the spatial union of valid pixels from all images.

    A zero-valued fallback guarantees a valid output when the
    period contains no observations.
    """
    valid_collection = (
        ee.ImageCollection(collection)
        .map(mask_function)
    )

    fallback = (
        ee.Image.constant(0)
        .rename("valid")
        .uint8()
    )

    return (
        valid_collection
        .merge(
            ee.ImageCollection(
                [fallback]
            )
        )
        .max()
        .rename("valid")
        .uint8()
    )


def calculate_area_coverage_pct(
    valid_union,
    geometry,
    scale_m,
):
    """
    Calculate area-weighted valid coverage over the footprint.
    """
    geometry = ee.Geometry(geometry)

    valid_area = (
        ee.Image.pixelArea()
        .multiply(
            ee.Image(valid_union)
        )
        .rename("valid_area")
    )

    covered_area_m2 = ee.Number(
        valid_area
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=geometry,
            scale=scale_m,
            maxPixels=1e7,
        )
        .get("valid_area")
    )

    footprint_area_m2 = (
        geometry.area(1)
    )

    return (
        covered_area_m2
        .divide(
            footprint_area_m2
        )
        .multiply(100)
    )

In [32]:
# ============================================================
# Period-level common optical coverage
# ============================================================

coverage_rows = []

station_list = (
    station_footprints
    .toList(
        station_footprints.size()
    )
)

station_count = (
    station_footprints
    .size()
    .getInfo()
)


for station_index in range(
    station_count
):

    footprint = ee.Feature(
        station_list.get(
            station_index
        )
    )

    station_name = (
        footprint
        .get("station")
        .getInfo()
    )

    geometry = (
        footprint.geometry()
    )

    print(
        "Processing:",
        station_name,
    )

    local_tiles = (
        get_local_mgrs_tiles(
            footprint
        )
    )

    local_s30 = (
        hls_s30_tiled
        .filter(
            ee.Filter.inList(
                "hls_mgrs_tile",
                local_tiles,
            )
        )
    )

    local_l30 = (
        hls_l30_tiled
        .filter(
            ee.Filter.inList(
                "hls_mgrs_tile",
                local_tiles,
            )
        )
    )

    station_features = []


    for (
        period_start,
        period_end,
    ) in periods:

        start_string = (
            period_start.isoformat()
        )

        end_string = (
            period_end.isoformat()
        )

        # ----------------------------------------------------
        # Sentinel-2
        # ----------------------------------------------------

        s2_period = (
            s2_collection
            .filterBounds(geometry)
            .filterDate(
                start_string,
                end_string,
            )
        )

        s2_union = (
            build_valid_union(
                s2_period,
                build_s2_common_valid_mask,
            )
        )

        s2_coverage = (
            calculate_area_coverage_pct(
                s2_union,
                geometry,
                20,
            )
        )

        # ----------------------------------------------------
        # HLS-S30
        # ----------------------------------------------------

        s30_period = (
            local_s30
            .filterDate(
                start_string,
                end_string,
            )
        )

        s30_union = (
            build_valid_union(
                s30_period,
                lambda image:
                    build_hls_common_valid_mask(
                        image,
                        sensor="S30",
                        exclude_high_aerosol=True,
                    ),
            )
        )

        s30_coverage = (
            calculate_area_coverage_pct(
                s30_union,
                geometry,
                30,
            )
        )

        # ----------------------------------------------------
        # HLS-L30
        # ----------------------------------------------------

        l30_period = (
            local_l30
            .filterDate(
                start_string,
                end_string,
            )
        )

        l30_union = (
            build_valid_union(
                l30_period,
                lambda image:
                    build_hls_common_valid_mask(
                        image,
                        sensor="L30",
                        exclude_high_aerosol=False,
                    ),
            )
        )

        l30_coverage = (
            calculate_area_coverage_pct(
                l30_union,
                geometry,
                30,
            )
        )

        # ----------------------------------------------------
        # HLS combined
        #
        # Union is performed at pixel level, not by simply
        # adding S30 and L30 date counts.
        # ----------------------------------------------------

        combined_union = (
            ee.ImageCollection(
                [
                    s30_union,
                    l30_union,
                ]
            )
            .max()
            .rename("valid")
        )

        combined_coverage = (
            calculate_area_coverage_pct(
                combined_union,
                geometry,
                30,
            )
        )

        # ----------------------------------------------------
        # One feature per source
        # ----------------------------------------------------

        source_values = [
            (
                "S2",
                20,
                s2_coverage,
            ),
            (
                "HLS_S30",
                30,
                s30_coverage,
            ),
            (
                "HLS_L30",
                30,
                l30_coverage,
            ),
            (
                "HLS_COMBINED",
                30,
                combined_coverage,
            ),
        ]

        for (
            source,
            scale_m,
            coverage,
        ) in source_values:

            station_features.append(
                ee.Feature(
                    None,
                    {
                        "station":
                            station_name,

                        "period_start":
                            start_string,

                        "period_end":
                            end_string,

                        "source":
                            source,

                        "scale_m":
                            scale_m,

                        "coverage_pct":
                            coverage,
                    },
                )
            )

    # --------------------------------------------------------
    # Evaluate one station at a time
    # --------------------------------------------------------

    station_collection = (
        ee.FeatureCollection(
            station_features
        )
    )

    station_info = (
        station_collection
        .getInfo()
    )

    station_rows = [
        feature["properties"]
        for feature
        in station_info["features"]
    ]

    coverage_rows.extend(
        station_rows
    )

    print(
        "  rows:",
        len(station_rows),
    )

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Processing: Pastos limpios
  rows: 552
Processing: Palma
  rows: 552
Processing: Bananera
  rows: 552
Processing: Manglar
  rows: 552
Processing: Bosque seco
  rows: 552


In [33]:
optical_period_coverage = pd.DataFrame(
    coverage_rows
)

optical_period_coverage[
    "coverage_pct"
] = pd.to_numeric(
    optical_period_coverage[
        "coverage_pct"
    ],
    errors="coerce",
)

print(
    "Rows:",
    len(optical_period_coverage),
)

print(
    "Expected:",
    5 * 138 * 4,
)

display(
    optical_period_coverage.head()
)

assert (
    len(optical_period_coverage)
    == 2760
)

assert (
    optical_period_coverage[
        "coverage_pct"
    ]
    .between(
        0,
        100.5,
    )
    .all()
)

Rows: 2760
Expected: 2760


,coverage_pct,period_end,period_start,scale_m,source,station
0,99.304466,2021-01-09,2021-01-01,20,S2,Pastos limpios
1,99.822055,2021-01-09,2021-01-01,30,HLS_S30,Pastos limpios
2,0.000000,2021-01-09,2021-01-01,30,HLS_L30,Pastos limpios
3,99.822055,2021-01-09,2021-01-01,30,HLS_COMBINED,Pastos limpios
4,99.304466,2021-01-17,2021-01-09,20,S2,Pastos limpios


In [34]:
coverage_thresholds = [
    0,
    50,
    70,
    80,
    90,
    99,
]


summary_rows = []


for (
    station,
    source,
), group in (
    optical_period_coverage
    .groupby(
        [
            "station",
            "source",
        ]
    )
):

    row = {
        "station": station,
        "source": source,
        "periods": len(group),
        "mean_coverage": (
            group["coverage_pct"].mean()
        ),
        "median_coverage": (
            group["coverage_pct"].median()
        ),
    }

    for threshold in coverage_thresholds:

        if threshold == 0:
            count = (
                group["coverage_pct"]
                .gt(0)
                .sum()
            )

            column_name = (
                "periods_gt_0"
            )

        else:
            count = (
                group["coverage_pct"]
                .ge(threshold)
                .sum()
            )

            column_name = (
                f"periods_ge_{threshold}"
            )

        row[column_name] = int(
            count
        )

    summary_rows.append(
        row
    )


coverage_summary = pd.DataFrame(
    summary_rows
)

display(
    coverage_summary
)

,station,source,periods,mean_coverage,median_coverage,periods_gt_0,periods_ge_50,periods_ge_70,periods_ge_80,periods_ge_90,periods_ge_99
0,Bananera,HLS_COMBINED,138,49.013911,52.851213,87,69,63,59,53,48
1,Bananera,HLS_L30,138,29.224186,0.000000,52,41,39,37,34,33
2,Bananera,HLS_S30,138,37.709612,0.004818,69,54,49,45,40,28
3,Bananera,S2,138,72.158393,99.288740,118,104,94,89,86,79
4,Bosque seco,HLS_COMBINED,138,64.838187,99.819940,103,89,86,85,83,79
5,Bosque seco,HLS_L30,138,39.593553,0.000000,61,54,53,53,50,50
6,Bosque seco,HLS_S30,138,53.551947,73.976419,91,75,70,67,62,57
7,Bosque seco,S2,138,79.743930,99.303323,120,113,108,106,102,95
8,Manglar,HLS_COMBINED,138,75.468691,99.020272,114,105,102,102,96,91
9,Manglar,HLS_L30,138,40.691397,0.000000,64,56,55,55,52,52


In [35]:
# ============================================================
# Temporal distribution of optical coverage
# ============================================================

temporal_coverage = (
    optical_period_coverage
    .copy()
)

temporal_coverage[
    "period_start"
] = pd.to_datetime(
    temporal_coverage[
        "period_start"
    ]
)

temporal_coverage[
    "year"
] = (
    temporal_coverage[
        "period_start"
    ]
    .dt.year
)

temporal_coverage[
    "month"
] = (
    temporal_coverage[
        "period_start"
    ]
    .dt.month
)


# Keep the two main candidates.
main_candidates = (
    temporal_coverage[
        temporal_coverage[
            "source"
        ].isin(
            [
                "S2",
                "HLS_COMBINED",
            ]
        )
    ]
    .copy()
)


# ============================================================
# Annual summary
# ============================================================

annual_summary = (
    main_candidates
    .groupby(
        [
            "source",
            "year",
        ],
        as_index=False,
    )
    .agg(
        periods=(
            "coverage_pct",
            "size",
        ),
        mean_coverage=(
            "coverage_pct",
            "mean",
        ),
        median_coverage=(
            "coverage_pct",
            "median",
        ),
        periods_ge_80=(
            "coverage_pct",
            lambda x: int(
                (x >= 80).sum()
            ),
        ),
        periods_ge_90=(
            "coverage_pct",
            lambda x: int(
                (x >= 90).sum()
            ),
        ),
        periods_ge_99=(
            "coverage_pct",
            lambda x: int(
                (x >= 99).sum()
            ),
        ),
    )
)

display(
    annual_summary
)


# ============================================================
# Monthly/climatological summary
# ============================================================

monthly_summary = (
    main_candidates
    .groupby(
        [
            "source",
            "month",
        ],
        as_index=False,
    )
    .agg(
        periods=(
            "coverage_pct",
            "size",
        ),
        mean_coverage=(
            "coverage_pct",
            "mean",
        ),
        median_coverage=(
            "coverage_pct",
            "median",
        ),
        periods_ge_80=(
            "coverage_pct",
            lambda x: int(
                (x >= 80).sum()
            ),
        ),
        periods_ge_90=(
            "coverage_pct",
            lambda x: int(
                (x >= 90).sum()
            ),
        ),
        periods_ge_99=(
            "coverage_pct",
            lambda x: int(
                (x >= 99).sum()
            ),
        ),
    )
)

display(
    monthly_summary
)

,source,year,periods,mean_coverage,median_coverage,periods_ge_80,periods_ge_90,periods_ge_99
0,HLS_COMBINED,2021,230,62.367794,98.065103,133,124,115
1,HLS_COMBINED,2022,230,49.219403,52.819627,101,95,91
2,HLS_COMBINED,2023,230,66.833023,99.020272,147,142,130
3,S2,2021,230,72.564360,99.295049,158,151,141
4,S2,2022,230,69.275297,99.288740,150,141,126
5,S2,2023,230,85.005593,99.301358,189,185,173


,source,month,periods,mean_coverage,median_coverage,periods_ge_80,periods_ge_90,periods_ge_99
0,HLS_COMBINED,1,60,95.641566,99.819940,56,56,52
1,HLS_COMBINED,2,60,98.486673,99.819940,60,56,52
2,HLS_COMBINED,3,60,54.881753,80.767842,31,27,25
3,HLS_COMBINED,4,45,42.516658,11.728224,16,14,13
4,HLS_COMBINED,5,60,48.049617,37.553136,24,23,21
5,HLS_COMBINED,6,60,43.342120,13.720982,22,21,19
6,HLS_COMBINED,7,60,65.311999,99.020272,36,33,32
7,HLS_COMBINED,8,60,37.122753,1.943151,18,17,15
8,HLS_COMBINED,9,60,43.493088,7.118997,25,23,21
9,HLS_COMBINED,10,45,32.217966,0.000000,12,12,10
